# Model And Feature Analysis

            This notebook summarizes model quality, feature importance, SHAP interpretation, and the auto-renew ablation experiment.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
MODELS = PROJECT_ROOT / "models"

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

## Model Leaderboard

In [ ]:
comparison = pd.read_csv(REPORTS / "model_comparison.csv")
display(comparison.sort_values("val_auc_pr", ascending=False))

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=comparison.sort_values("val_auc_pr", ascending=False), x="val_auc_pr", y="model", color="#4c78a8")
ax.set_title("Validation AUC-PR by model")
ax.set_xlabel("Validation AUC-PR")
ax.set_ylabel("")
plt.show()

## Held-Out Test Evaluation

In [ ]:
test_eval = pd.read_csv(REPORTS / "test_evaluation.csv")
display(test_eval)

## Feature Importance

In [ ]:
importance = pd.read_csv(REPORTS / "feature_importances.csv")
top_by_model = (
    importance.sort_values(["model", "importance_norm"], ascending=[True, False])
    .groupby("model")
    .head(15)
)
display(top_by_model)

for model_name, group in top_by_model.groupby("model"):
    plt.figure(figsize=(9, 5))
    ax = sns.barplot(data=group.sort_values("importance_norm"), x="importance_norm", y="feature", color="#72b7b2")
    ax.set_title(f"Top feature importances: {model_name}")
    ax.set_xlabel("Normalized importance")
    ax.set_ylabel("")
    plt.show()

## SHAP Summary

In [ ]:
shap_summary_path = REPORTS / "shap_summary.csv"
if shap_summary_path.exists():
    shap_summary = pd.read_csv(shap_summary_path)
    display(shap_summary.head(25))

    plt.figure(figsize=(9, 6))
    top_shap = shap_summary.head(20).sort_values("mean_abs_shap")
    ax = sns.barplot(data=top_shap, x="mean_abs_shap", y="feature", color="#e45756")
    ax.set_title("Top SHAP features")
    ax.set_xlabel("Mean absolute SHAP value")
    ax.set_ylabel("")
    plt.show()

## Saved SHAP Figures

In [ ]:
from IPython.display import Image, display

for image_name in ["shap_top20.png", "shap_beeswarm.png"]:
    path = REPORTS / image_name
    if path.exists():
        print(path)
        display(Image(filename=str(path)))

## Auto-Renew Ablation

In [ ]:
ablation_path = REPORTS / "auto_renew_ablation.csv"
if ablation_path.exists():
    ablation = pd.read_csv(ablation_path)
    display(ablation)

    val_ablation = ablation[ablation["split"] == "val"].copy()
    baseline = comparison.loc[comparison["model"] == "xgboost", [
        "val_auc_pr", "val_recall", "val_precision", "val_lift_at_top10"
    ]].iloc[0]
    summary = pd.DataFrame([
        {
            "experiment": "xgboost_baseline",
            "auc_pr": baseline["val_auc_pr"],
            "recall": baseline["val_recall"],
            "precision": baseline["val_precision"],
            "lift_at_top10": baseline["val_lift_at_top10"],
        },
        {
            "experiment": "drop_auto_renew_cancel_family",
            "auc_pr": val_ablation["auc_pr"].iloc[0],
            "recall": val_ablation["recall"].iloc[0],
            "precision": val_ablation["precision"].iloc[0],
            "lift_at_top10": val_ablation["lift_at_top_fraction"].iloc[0],
        },
    ])
    display(summary)

    ax = sns.barplot(data=summary, x="auc_pr", y="experiment", color="#54a24b")
    ax.set_title("Ablation impact on validation AUC-PR")
    ax.set_xlabel("Validation AUC-PR")
    ax.set_ylabel("")
    plt.show()

## Interpretation

            - `auto_renew_rate` is an expected high-signal subscription feature, not automatically a leakage bug.
            - The ablation confirms renewal/cancel features are materially useful, but the model still ranks churn risk without them.
            - Newly engineered recency and momentum features should be monitored after retraining to confirm attribution becomes more distributed.